In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from fmlib.utils.mlstorage.functions import save_model_to_mlstorage

Актуальные URL для MLStorage можно найти в его UI, адреса которого в разных контурах перечислены [здесь](https://confluence.sberbank.ru/pages/viewpage.action?pageId=14598378959)

В качестве логина и токена требуется логин и пароль УЗ в выбранном домене или логин и токен ТУЗ (присутствуют в окружении в ЛД и Airflow Sigma)

Сертификаты не требуется проставлять в ЛД и Airflow Sigma. Инструкция по созданию файла с сертификатами:

- для Sigma [здесь](https://confluence.sberbank.ru/pages/viewpage.action?pageId=6666847551) выбираете сертификат **sberca-ext**. возьмите два эти сертификата: промежуточный и корневой, сохраните в один файл (сначала root, потом обычный) и используйте путь к файлу для параметра verify

- для Alpha [здесь](https://confluence.sberbank.ru/pages/viewpage.action?pageId=6666847551) выбираете сертификат **sberca-int**. возьмите два эти сертификата: промежуточный и корневой, сохраните в один файл (сначала root, потом обычный) и используйте путь к файлу для параметра verify

In [3]:
network = "omega"
if network == "sigma":
    os.environ["ML_STORAGE_URL"] = (
        "https://api-adapter-s3.ci02741861-epromgen1dbz-mls-minio.apps.prom-gen1-dbz.sigma.sbrf.ru/s3"
    )
elif network == "omega":
    os.environ["ML_STORAGE_URL"] = "https://s3.mls.iaz.omega.sbrf.ru/s3"

os.environ["ML_STORAGE_LOGIN"] = "21582431"  # User's domain ID / TUZ LOGIN

os.environ["ML_STORAGE_VERIFY"] = "certs.pem"

In [4]:
import getpass

os.environ["ML_STORAGE_TOKEN"] = getpass.getpass("PUZ Password:")

### Save model

**Рекомендации по сохранению**
- Файлы с весами модели должны иметь формат `.safetensors`
- Файлы с конфигурацией должна иметь формат `.yaml`
- Для того, чтобы не запутаться в версиях модели, рекомендуется дополнительно сохранять простой `.txt` файл с описанием модели:
  - Когда модель натренирована
  - Кем натренирована
  - Какие данные использовались
  - Версия библиотеки, использованная для тренировки
  - Дополнительные рекомендации по использованию
- Полный список поддерживаемых форматов с пояснениями доступен [в документации сервиса](https://confluence.sberbank.ru/pages/viewpage.action?pageId=16966878053)
- Логика работы `model_id` и `model_version` подробнее описана в `fmlib.utils.mlstorage.functions.generate_mlstorage_path`

In [5]:
model_files = [
    "/home/datalab/nfs/checkpoints/test_qini/checkpoint_000005/model.safetensors",
    "/home/datalab/nfs/checkpoints/test_qini/checkpoint_000005/config.yaml",
    "/home/datalab/nfs/checkpoints/test_qini/checkpoint_000005/metrics.yaml",
]

save_model_to_mlstorage(
    model_files=model_files,
    model_id="302621/303111",
    model_version="0.1.*",
)

### Read model

In [6]:
from fmlib.utils.loading.load_pipeline import load_inference_pipeline_from_mlstorage

- Логика работы `model_id` и `model_version` подробнее описана в `fmlib.utils.mlstorage.functions.generate_mlstorage_path`

In [7]:
load_inference_pipeline_from_mlstorage(
    model_id="302621/303111",
    model_version="0.1.15",
)

InferencePipeline(
  (model): UpliftFeatureTransformer(
    (transformer): FeatureTransformer(
      (transformer_blocks): ModuleList(
        (0-2): 3 x EncoderBlock(
          (multi_head_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (attention_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (attention_dropout): Dropout(p=0.15, inplace=False)
          (ffn): STEv2FFN(
            (w1): Linear(in_features=64, out_features=256, bias=True)
            (act): SELU()
            (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.15, inplace=False)
            (w2): Linear(in_features=256, out_features=64, bias=True)
          )
          (ffn_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.15, inplace=False)
        )
      )
      (agg_layer): LinearAggregation(
        (agg